# Week 8 · Day 2 — Unit economics of a loan: starter

Bekzod's question: what does one loan earn after the money, the losses and the cost of selling it?
The assumptions are in **one cell** — change them there and every number below follows. Write `unit_economics` (TODO).

In [1]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DATA = Path("../..")                      # this notebook lives in data/python/week8/, so "../.." is the data/ folder
con = sqlite3.connect(DATA / "raw" / "nur.db")
ASOF = "2026-10-01"
sql = lambda q: pd.read_sql(q, con)       # sql("SELECT ...") returns a DataFrame
print(sql("SELECT COUNT(*) AS loans FROM loans"))

   loans
0   7828


In [2]:
# The course's clean layer (week 3 Day 4), created as TEMP views: they live only in this connection, nur.db is not changed.
con.executescript("""
CREATE TEMP VIEW IF NOT EXISTS v_payments_clean AS
WITH ranked AS (
  SELECT payment_id, loan_id, installment_id, paid_at, amount,
         REPLACE(LOWER(TRIM(method)), 'bank transfer', 'transfer') AS method,
         ROW_NUMBER() OVER (PARTITION BY installment_id, paid_at, amount ORDER BY payment_id) AS rn
  FROM payments)
SELECT payment_id, loan_id, installment_id, paid_at, amount, method FROM ranked WHERE rn = 1;
CREATE TEMP VIEW IF NOT EXISTS v_loan_status AS
WITH unpaid AS (
  SELECT i.loan_id, i.due_date FROM installments i
  LEFT JOIN v_payments_clean p ON p.installment_id = i.installment_id AND p.paid_at <= '2026-10-01'
  WHERE i.due_date <= '2026-10-01'
  GROUP BY i.installment_id, i.loan_id, i.due_date, i.total_due
  HAVING COALESCE(SUM(p.amount), 0) < i.total_due),
dpd AS (SELECT loan_id, CAST(julianday('2026-10-01') - julianday(MIN(due_date)) AS INTEGER) AS dpd FROM unpaid GROUP BY loan_id),
outst AS (SELECT loan_id, SUM(principal_due) AS outstanding FROM installments WHERE due_date > '2026-10-01' GROUP BY loan_id)
SELECT l.loan_id, l.customer_id, p.code AS product, p.channel, b.region, l.risk_grade, l.status, l.disbursed_at, l.principal,
       l.term_months, COALESCE(o.outstanding, 0) AS outstanding, COALESCE(d.dpd, 0) AS dpd,
       CASE WHEN COALESCE(d.dpd, 0) = 0 THEN '0' WHEN d.dpd <= 30 THEN '1-30' WHEN d.dpd <= 60 THEN '31-60'
            WHEN d.dpd <= 90 THEN '61-90' ELSE '90+' END AS bucket
FROM loans l JOIN products p ON p.product_id = l.product_id JOIN branches b ON b.branch_id = l.branch_id
LEFT JOIN dpd d ON d.loan_id = l.loan_id LEFT JOIN outst o ON o.loan_id = l.loan_id;
""")
print(sql("SELECT COUNT(*) AS loans, SUM(outstanding) AS outstanding FROM v_loan_status"))

   loans   outstanding
0   7828  208723504430


## 0. Assumptions (ours — write them under every table you show)

In [3]:
COF = 0.18                                               # funding rate, per year
LGD = 0.60                                               # loss given default
ACQ = {"app": 60_000, "branch": 450_000, "partner": 100_000}    # acquisition cost per loan, soum
SERV = {"app": 8_000, "branch": 25_000, "partner": 8_000}       # servicing cost per month of the schedule, soum
REPEAT_ACQ = 20_000                                      # selling a repeat loan to a known customer

In [4]:
inst = sql("SELECT loan_id, installment_no, due_date, principal_due, interest_due FROM installments ORDER BY loan_id, installment_no")
ls = sql("SELECT loan_id, product, channel, principal, term_months, dpd FROM v_loan_status")
print(len(inst), "instalments ·", len(ls), "loans")

92193 instalments · 7828 loans


## 1. Walk-through — one loan (the page's worked example, loan 1)

In [5]:
one = inst[inst["loan_id"] == 1].copy()
one["bal_before"] = 31_150_000 - (one["principal_due"].cumsum() - one["principal_due"])
I, cof = one["interest_due"].sum(), one["bal_before"].sum() * COF / 12
print(f"interest over the schedule {I:,.0f} · balance-months {one['bal_before'].sum():,.0f} · cost of funds {cof:,.0f}")

interest over the schedule 10,650,293 · balance-months 426,011,732 · cost of funds 6,390,176


## 2. TODO — `unit_economics(inst, ls)`
Per loan: `interest` (Σ interest_due), `cof` (Σ balance before each instalment × COF / 12).
Per product: `pd` = share of its loans with dpd > 90; `ead_ratio` = mean, over its 90+ loans, of (principal not yet
repaid at the first unpaid instalment ÷ principal) — hint: the balance before the first instalment that is unpaid on the
as-of date; then per product the averages and `income = (1 − pd) × interest`, `el = pd × LGD × ead_ratio × principal`,
`acq`, `serv = SERV × term`, `net`, `net_per_1000`.

In [6]:
def unit_economics(inst, ls):
    """DataFrame indexed by product code with at least: loans, principal, term, interest, cof, pd, ead_ratio, income, el,
    acq, serv, net, net_per_1000 (averages per loan)."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [7]:
ue = unit_economics(inst, ls)
assert set(ue.index) == {"AUTO", "BNPL", "CONS", "MICRO", "SME"}
assert ue["loans"].sum() == len(ls)
assert ue.loc["BNPL", "interest"] == 0, "BNPL charges the customer 0 %"
assert np.allclose(ue["net"], ue["income"] - ue["cof"] - ue["el"] - ue["acq"] - ue["serv"])
assert np.allclose(ue["net_per_1000"], 1000 * ue["net"] / ue["principal"])
print("unit_economics OK")

NotImplementedError: TODO: write me, then run the assert cell below

## P1

Average principal, term, interest and cost of funds per product; CoF ÷ interest.

In [8]:
# your code

## P2

PD, EAD ratio, EAD and EL per product.

In [9]:
# your code

## P3

Net per loan and per 1 000 soum lent; 2026 contribution per product.

In [10]:
# your code

## P4

BNPL: the merchant fee (% of principal) that makes it break even.

In [11]:
# your code

## P5

Micro sensitivity: net at PD 0/1/5/8 %; break-even PD (first and repeat loan); in FPD terms two ways.

In [12]:
# your code

## P6

Net per 1 000 at funding rates 14 / 18 / 22 %.

In [13]:
# your code

## P7

PD on matured loans only; net per 1 000 again.

In [14]:
# your code

## P8

LTV/CAC for Micro-first customers (first loan before 2024-10-01, 24 months).

In [15]:
# your code

## P10 (stretch)

Consumer loans by risk grade.

In [16]:
# your code